# Método: expectativa pitagórica en la ACB (8 temporadas)

Reproduce las cifras del artículo llamando al paquete `pitagorica`. Reglas en `docs/preregistro_v2.md`. Aquí se usan menos remuestreos (2000) que en `make analisis` (10.000), así que los intervalos varían algo.

In [1]:
import numpy as np

from pitagorica import SEMILLA, datos
from pitagorica import multitemporada as mt

rng = np.random.default_rng(SEMILLA)
df_all = datos.cargar_todas()
df = df_all[df_all["season"].isin(datos.temporadas_usables(df_all))]
ts = datos.tabla_temporadas(df)
print(len(ts), "equipos-temporada de", ts["club"].nunique(), "clubes")
ts.head()

145 equipos-temporada de 24 clubes


,season,club,G,V,PF,PC,pos,G_ig,V_ig,pct_v,net_rtg,pct_ig
0,2017_18,Andorra,34,19,2898,2787,2632.96,11,3,0.558824,4.215788,0.272727
1,2017_18,Barça,34,24,3030,2691,2586.80,13,7,0.705882,13.104995,0.538462
2,2017_18,Baskonia,34,25,2946,2668,2568.34,11,7,0.735294,10.824112,0.636364
3,2017_18,Betis,34,7,2629,3096,2582.38,10,4,0.205882,-18.084093,0.400000
4,2017_18,Bilbao,34,8,2592,2843,2498.90,9,3,0.235294,-10.044420,0.333333


## Calibrar k fuera de muestra

In [2]:
val = mt.validacion_k(ts)
print("k conjunto =", round(mt.k_conjunto(ts), 2))
print(
    "error medio k_sin_temporada:",
    round(val["error_k_sin_temporada"].mean(), 2),
    "| con k=14:",
    round(val["error_k14"].mean(), 2),
)
val.round(2)

k conjunto = 10.66
error medio k_sin_temporada: 1.53 | con k=14: 1.81


,temporada,k_sin_temporada,k_propio,error_k_sin_temporada,error_k14,error_k_propio
0,2017_18,10.91,9.23,1.55,2.22,1.45
1,2018_19,10.72,10.27,1.41,1.84,1.43
2,2020_21,10.48,12.04,1.46,1.63,1.39
3,2021_22,10.69,10.42,1.46,1.71,1.48
4,2022_23,10.87,9.66,1.40,2.25,1.38
5,2023_24,10.50,12.16,1.57,1.38,1.37
6,2024_25,10.58,11.28,1.78,1.93,1.76
7,2025_26,10.57,11.27,1.59,1.55,1.51


## Desvíos y pares t → t+1

In [3]:
td = mt.con_desvios(ts)
p = mt.pares(td)
print(len(p), "pares")
td[td["season"] == "2025_26"].sort_values("desvio34", ascending=False)[
    ["club", "V", "esperadas34", "desvio34"]
].round(2)

99 pares


,club,V,esperadas34,desvio34
138,Manresa,16,12.67,3.33
130,Bilbao,19,15.76,3.24
140,Real Madrid,26,24.07,1.93
137,Lleida,12,10.13,1.87
129,Baskonia,25,23.34,1.66
139,Murcia,25,23.73,1.27
136,Joventut,22,21.34,0.66
133,Girona,14,13.68,0.32
128,Barça,24,23.70,0.30
131,Breogán,15,14.80,0.20


## ¿Se repite? Test principal pre-registrado

In [4]:
res = mt.persistencia(p, 2000, rng)
print({k: res[k] for k in ("r", "pendiente", "n_pares", "conclusion")})

{'r': {'estimacion': 0.05397865551476154, 'ic90_bajo': -0.110418239041045, 'ic90_alto': 0.1985987140658847}, 'pendiente': {'estimacion': 0.05100197266009265, 'ic90_bajo': -0.11329908018837645, 'ic90_alto': 0.17313983825551618}, 'n_pares': 99, 'conclusion': 'compatible_con_azar'}
